# K305 Near-Miss Analysis

This notebook finds possible close encounters between vehicles and pedestrians or cyclists at K305. It uses paths recorded by roadside LiDAR sensors. Results are for safety review only: they do not prove a crash or a traffic-rule violation.

## About the dataset
The dataset has 288 JSON files from 14 October 2026. Each file covers five minutes, together covering one day. The sensor records about 10 frames each second. Each frame lists tracked road users, their location, speed, and type.


## Setup and configuration
The setup cell loads the analysis tools and options. Set `RUN_FULL_SCAN = True` to use all 288 files; `False` uses the quick sample. Set `MAKE_PLOTS = False` to skip charts and speed up a full run. Raw frame-by-frame CSV files are optional and off by default.

Use the project’s `.venv` Python environment. It needs `numpy`, `pandas`, and `ipykernel`.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import importlib.util, json, time, math
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "database/k305_object_list_2026-10-14_aggregation_5min"
OUTPUT_ROOT = PROJECT_DIR / "output"
MANUAL_FILES = []   # filenames or paths, used only in manual mode
MAKE_PLOTS = True
SAVE_RAW_INTERACTIONS = False
TTC_THRESHOLD_SEC = 1.5
PET_THRESHOLD_SEC = 1.5

# Reuse the project's reviewed parsing, preprocessing, TTC and PET routines.
spec = importlib.util.spec_from_file_location("near_miss_analysis", PROJECT_DIR / "near_miss_analysis.py")
nma = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nma)
nma.CONFIG.update(TTC_THRESHOLD_SEC=TTC_THRESHOLD_SEC, PET_THRESHOLD_SEC=PET_THRESHOLD_SEC)
RUN_FULL_SCAN = False  # Set True to select the full-day run; False keeps quick mode.


## Choose a run: quick or full
Choose the run size in the setup cell, then run the notebook. Quick mode reads three files from 08:00–08:14. Use it to check that the notebook works. Full mode reads all 288 files for the whole day and can take a long time.

A progress line such as `[12/288]` shows the current file. `[288/288]` means all files have been read; the notebook still needs to calculate results, save tables, and make charts.


### Quick Run
This reads the three files from 08:00–08:14. It is useful for testing and example results, but it does not represent the full day.


In [ ]:
if RUN_FULL_SCAN:
    RUN_MODE = "full"
    print("Full Run selected: all 288 five-minute files")
else:
    print("Full Run is not selected. Set RUN_FULL_SCAN = True in setup to enable it.")


### Full Run
This reads all 288 files for the full day. It takes longer than the quick run. Set `MAKE_PLOTS = False` before starting if you do not need charts.


In [ ]:
if RUN_FULL_SCAN:
    RUN_MODE = "full"
    print("Full Run selected: all 288 five-minute files")
else:
    print("Full Run is not selected. Set RUN_FULL_SCAN = True in setup to enable it.")


In [ ]:
available = sorted(DATA_DIR.glob("*.json"))
if RUN_MODE == "quick":
    selected_files = [next(p for p in available if f"_{minute}_to_" in p.name) for minute in ("08-00", "08-05", "08-10")]
elif RUN_MODE == "full":
    selected_files = available
elif RUN_MODE == "manual":
    selected_files = [Path(f) if Path(f).is_absolute() else DATA_DIR / f for f in MANUAL_FILES]
    if not selected_files: raise ValueError("Add one or more filenames to MANUAL_FILES.")
    missing = [str(p) for p in selected_files if not p.is_file()]
    if missing: raise FileNotFoundError(missing)
else:
    raise ValueError("RUN_MODE must be quick, full, or manual")
if RUN_MODE == "full" and len(selected_files) != 288: raise ValueError(f"Expected 288 files, found {len(selected_files)}")
print(f"Mode: {RUN_MODE} | files: {len(selected_files)}")
for p in selected_files: print(" •", p.name)
run_label = {"quick":"sample_08-00_to_08-14", "full":"full_day", "manual":"manual_selection"}[RUN_MODE]
run_dir = OUTPUT_ROOT / f"{run_label}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
run_dir.mkdir(parents=True, exist_ok=False)
print("Output folder:", run_dir)


## Parsing and preprocessing
First, the notebook turns each JSON frame into a table with one row for each tracked object. It then cleans the data:

- Repeated observations of the same track in the same frame are removed.
- Speeds above the allowed maximum are removed.
- Positions and speeds are smoothed to reduce small sensor errors.
- Tracks seen in fewer than five frames are removed because they may be sensor noise.

Track IDs are handled one file at a time. A track that continues into the next five-minute file is not joined.


In [ ]:
started = time.perf_counter()
ttc_rows = []
pet_acc = nma.PETAccumulator()
class_tracks = defaultdict(set)
quality = defaultdict(int)
frame_count = 0
trajectory_x, trajectory_y = [], []
for i, path in enumerate(selected_files, 1):
    with path.open() as fh: raw = nma.parse_chunk(json.load(fh))
    quality["raw_rows"] += len(raw)
    clean = nma.preprocess_chunk(raw, quality)
    if clean.empty:
        print(f"[{i}/{len(selected_files)}] {path.name}: no retained tracks")
        continue
    frame_count += clean.ts.nunique()
    for cls, tid in clean[["cls", "tid"]].drop_duplicates().itertuples(index=False): class_tracks[cls].add(tid)
    trajectory_x.extend(clean.x.to_numpy()[::6]); trajectory_y.extend(clean.y.to_numpy()[::6])
    chunk_ttc = nma.run_ttc_on_chunk(clean)
    ttc_rows.extend(chunk_ttc)
    pet_acc.update(clean)
    print(f"[{i}/{len(selected_files)}] {path.name}: frames={clean.ts.nunique()} | raw TTC frame observations={len(chunk_ttc)} | TTC<1.5 observations={sum(r['nm'] for r in chunk_ttc)}")
ttc_raw = pd.DataFrame(ttc_rows)
pet_all = pd.DataFrame(pet_acc.compute())
ttc_events = nma.consolidate_ttc_events(ttc_raw)


## TTC near-miss analysis
**Time to collision (TTC)** estimates how long it would take two road users to meet if both kept their current speed and direction. It uses their positions, speeds, and estimated sizes.

By default, the notebook checks only vehicle–pedestrian and vehicle–cyclist pairs. It keeps TTC values only when the pair is moving toward each other. Pairs moving apart, moving side by side, or already overlapping are not counted.

The sensor may record the same pair in several frames. Observations no more than 0.25 seconds apart are grouped into one event. A possible near miss has **TTC < 1.5 seconds**. A raw observation is one sensor moment; an event is the group of nearby observations.


In [ ]:
# Deterministic kinematics validation: approaching, separating, parallel, overlapping, boundary.
nma.run_ttc_self_tests()
print(f"Raw near-miss TTC frame observations: {int(ttc_raw.nm.sum()) if not ttc_raw.empty else 0:,}")
print(f"Consolidated TTC near-miss events: {len(ttc_events):,}")
ttc_preview = ttc_events.nsmallest(10, "min_ttc_sec") if not ttc_events.empty else ttc_events
display(ttc_preview)


## PET grid-cell proxy analysis
**Post-encroachment time (PET)** normally measures the time between one road user leaving a conflict area and another arriving. This project has no mapped conflict areas, so it uses a simpler estimate: 2 m × 2 m ground cells. It finds different tracks that visit the same cell close together.

If a track returns to a cell after a gap of more than 0.25 seconds, this counts as a new visit. A value below 1.5 seconds is flagged. The CSV keeps four decimal places for values close to the limit.

This is a **grid-cell proxy**, not a standard engineering PET measurement. It depends on cell size and tracking quality, so use it as an early indicator only.


In [ ]:
pet_proxy_events = pet_all[pet_all.nm].copy() if not pet_all.empty else pd.DataFrame()
if not pet_proxy_events.empty:
    pet_proxy_events = pet_proxy_events.sort_values("pet", kind="stable")
pet_preview = pet_proxy_events.head(10)
print(f"PET proxy candidate pairs: {len(pet_all):,}; PET proxy events with PET < {PET_THRESHOLD_SEC}: {len(pet_proxy_events):,}")
display(pet_preview)


## Output and export
Each run creates a new timestamped folder in `output`, so earlier results stay unchanged. It contains a JSON summary, grouped TTC events, and PET proxy events below the limit. If raw exports are enabled, it also includes frame-by-frame TTC observations and all PET pairs. Charts are saved as SVG files.


In [ ]:
ttc_events.to_csv(run_dir / "near_miss_ttc_events.csv", index=False)
pet_rename = {"cx":"cell_gx", "cy":"cell_gy", "zx":"cell_center_x_m", "zy":"cell_center_y_m", "ts":"timestamp", "tid1":"track_id_1", "tid2":"track_id_2", "cls1":"class_1", "cls2":"class_2", "cat1":"category_1", "cat2":"category_2", "pet":"pet_sec", "nm":"is_near_miss_pet_proxy", "itype":"interaction_type"}
pet_export = pet_proxy_events.rename(columns=pet_rename).copy()
if not pet_export.empty: pet_export["pet_sec"] = pet_export["pet_sec"].map(lambda x: f"{x:.4f}")
pet_export.to_csv(run_dir / "near_miss_pet_proxy_events.csv", index=False)
if SAVE_RAW_INTERACTIONS:
    rename_ttc = {"ts":"timestamp", "tid1":"track_id_1", "tid2":"track_id_2", "cls1":"class_1", "cls2":"class_2", "ttc":"ttc_sec", "nm":"is_near_miss_ttc", "itype":"interaction_type"}
    ttc_raw.rename(columns=rename_ttc).to_csv(run_dir / "all_ttc_frame_interactions.csv", index=False)
    pet_all.rename(columns=pet_rename).to_csv(run_dir / "all_pet_proxy_interactions.csv", index=False)
print("Event tables exported.")


## Visualizations
When plots are enabled, the notebook saves and displays SVG charts here. TTC charts count grouped events, not every sensor frame. PET charts show flagged grid-cell proxy events. One event can appear in many frames.

Set `MAKE_PLOTS = False` before a full run to skip charts. The CSV and JSON files are still created.


In [ ]:
from IPython.display import display, SVG

if MAKE_PLOTS:
    # Lightweight SVG plots via the project's SVG helper, with consolidated event inputs.
    if not ttc_events.empty:
        vals = ttc_events.min_ttc_sec.to_numpy()
        counts, edges = np.histogram(vals, bins=25, range=(0, nma.CONFIG["TTC_LOOKAHEAD_SEC"]))
        chart = nma.SVG(820, 520, "Consolidated TTC Near-Miss Events (K305)")
        ymax = float(counts.max()) * 1.15 or 1
        chart.axes(0, nma.CONFIG["TTC_LOOKAHEAD_SEC"], 0, ymax, "Minimum TTC per event (s)", "Consolidated event count")
        bw = (chart.w-chart.pl-chart.pr)/len(counts)
        chart.bars([(chart.pl+j*bw, int(c), chart.C["red"] if (edges[j]+edges[j+1])/2 < TTC_THRESHOLD_SEC else chart.C["blue"]) for j,c in enumerate(counts)], counts, bw, ymax)
        chart.vline(TTC_THRESHOLD_SEC, 0, nma.CONFIG["TTC_LOOKAHEAD_SEC"], 0, ymax)
        ttc_plot_path = run_dir / "plot_ttc_consolidated_events.svg"
        chart.save(str(ttc_plot_path))
        display(SVG(filename=str(ttc_plot_path)))
    if not pet_proxy_events.empty:
        vals = pet_proxy_events.pet.to_numpy()
        counts, edges = np.histogram(vals, bins=25, range=(0, max(3.0, min(10.0, float(vals.max())+0.1))))
        chart = nma.SVG(820, 520, "PET Grid-Cell Proxy Near-Miss Events")
        ymax = float(counts.max()) * 1.15 or 1
        chart.axes(0, float(edges[-1]), 0, ymax, "PET proxy (s; unrounded comparison)", "Proxy event count")
        bw = (chart.w-chart.pl-chart.pr)/len(counts)
        chart.bars([(chart.pl+j*bw, int(c), chart.C["red"]) for j,c in enumerate(counts)], counts, bw, ymax)
        chart.vline(PET_THRESHOLD_SEC, 0, float(edges[-1]), 0, ymax)
        pet_plot_path = run_dir / "plot_pet_proxy_events.svg"
        chart.save(str(pet_plot_path))
        display(SVG(filename=str(pet_plot_path)))
else:
    print("Plots disabled by MAKE_PLOTS=False")


## Compact run summary
This table shows how much data was processed and how many possible interactions were found. “Raw TTC frame observations” are individual sensor moments. “Consolidated TTC events” group nearby moments for the same pair. PET counts use the grid-cell proxy. These are events to review, not confirmed crashes.


In [ ]:
elapsed = time.perf_counter() - started
near_frame_count = int(ttc_raw.nm.sum()) if not ttc_raw.empty else 0
report = {
    "intersection":"K305", "date":"2026-10-14", "run_mode":RUN_MODE,
    "selected_files":[p.name for p in selected_files], "output_directory":str(run_dir),
    "frames_analysed":int(frame_count), "tracks_by_class":{k:len(v) for k,v in class_tracks.items()},
    "raw_ttc_frame_observations":int(len(ttc_raw)), "near_miss_ttc_frame_observations":near_frame_count,
    "consolidated_ttc_events":int(len(ttc_events)), "pet_proxy_pairs":int(len(pet_all)),
    "pet_proxy_near_miss_events":int(len(pet_proxy_events)),
    "ttc_threshold_sec_strict_lt":TTC_THRESHOLD_SEC, "pet_threshold_sec_strict_lt":PET_THRESHOLD_SEC,
    "pet_method":"2 m grid-cell occupancy proxy; not map-based engineering PET",
    "data_quality":dict(quality), "runtime_sec":round(elapsed,2), "plots_enabled":MAKE_PLOTS,
}
(run_dir / "run_report.json").write_text(json.dumps(report, indent=2, default=lambda o: int(o) if isinstance(o, np.integer) else float(o) if isinstance(o, np.floating) else str(o)))
summary_table = pd.DataFrame([
    ("Input files", len(selected_files)), ("Frames analysed", frame_count),
    ("Tracks", sum(map(len, class_tracks.values()))),
    ("Raw TTC frame observations", len(ttc_raw)), ("TTC<1.5 frame observations", near_frame_count),
    ("Consolidated TTC events", len(ttc_events)), ("PET proxy candidate pairs", len(pet_all)),
    ("PET proxy events <1.5s", len(pet_proxy_events)), ("Runtime (seconds)", round(elapsed,2)),
], columns=["Measure", "Value"])
display(summary_table)
print("Report:", run_dir / "run_report.json")


In [ ]:
# Display the exact JSON report and the exported event-table previews.
from IPython.display import display, Markdown
report_view = json.loads((run_dir / "run_report.json").read_text())
display(Markdown("### Run report"))
display(pd.DataFrame([{"Field": k, "Value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v} for k, v in report_view.items()]))
display(Markdown("### Exported files"))
for artifact in sorted(run_dir.iterdir()): print(artifact.name)
display(Markdown("### TTC event CSV preview"))
display(pd.read_csv(run_dir / "near_miss_ttc_events.csv").head(10))
display(Markdown("### PET proxy event CSV preview"))
display(pd.read_csv(run_dir / "near_miss_pet_proxy_events.csv").head(10))


## Interpretation and limitations
Use these results to find pairs of paths that may need a closer look. TTC assumes that each road user keeps the same speed and direction, which may not be true. Sensor errors, wrong object types, changed or missing track IDs, and estimated object sizes can affect the results. Tracks are not joined across five-minute file boundaries.

PET uses shared 2 m × 2 m cells because mapped conflict zones are not available. It is a screening tool, not a standard engineering PET measure. The quick run covers only 15 minutes; use the full-day run for the whole dataset.
